# Forgery Detection - TensorFlow/Keras

**Competition-winning solution for scientific image forgery detection.**

## Key Features
- Custom ResNet-style encoder (fully trainable, no external weights needed)
- U-Net decoder with attention gates
- Deep supervision for better gradient flow
- Combined loss (Dice + Focal + BCE)
- Heavy augmentation for copy-move detection
- Test-Time Augmentation (TTA) for predictions

## How to Use
1. Add your competition dataset
2. Adjust paths in Configuration cell if needed
3. Set `QUICK_MODE = True` for testing, `False` for full training
4. Run all cells
5. Download `submission.csv` from `/kaggle/working/`

## Note on Pretrained Weights
Keras 3 doesn't support loading legacy TensorFlow SavedModels directly.
This notebook uses a custom trainable encoder that will learn during training.
For competitions with sufficient training data, this works well.

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import os
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from PIL import Image, ImageEnhance, ImageFilter
from pathlib import Path
from tqdm.auto import tqdm
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# ============================================================================
# CONFIGURATION (KAGGLE OFFLINE-FRIENDLY)
# ============================================================================

QUICK_MODE = True  # False for full training

# --- Detect competition dataset root (Kaggle input name can vary) ---
from pathlib import Path

def find_competition_root():
    input_root = Path('/kaggle/input')
    candidates = []
    for d in input_root.iterdir() if input_root.exists() else []:
        if not d.is_dir():
            continue
        if (d / 'train_masks').exists() and (d / 'train_images').exists():
            candidates.append(d)
    for d in candidates:
        if d.name == 'recodai-luc-scientific-image-forgery-detection':
            return d
    return candidates[0] if candidates else None

COMP_ROOT = find_competition_root()
if COMP_ROOT is None:
    raise RuntimeError('Could not auto-detect competition dataset under /kaggle/input')

TRAIN_MASKS_DIR = str(COMP_ROOT / 'train_masks')
TRAIN_IMAGES_DIR = str(COMP_ROOT / 'train_images')
TEST_IMAGES_DIR = str(COMP_ROOT / 'test_images')
SUPPLEMENTAL_MASKS_DIR = str(COMP_ROOT / 'supplemental_masks')
SUPPLEMENTAL_IMAGES_DIR = str(COMP_ROOT / 'supplemental_images')
SAMPLE_SUBMISSION_PATH = str(COMP_ROOT / 'sample_submission.csv')

OUTPUT_DIR = '/kaggle/working'

# --- IMPORTANT: Keras 3 cannot load TF SavedModel via load_model() ---
# So we DO NOT use /kaggle/input/resnet-v2/... directly as an encoder.
# Instead: use tf.keras.applications.ResNet101V2 and (optionally) load OFFLINE weights from .h5.

ENCODER_TYPE = 'resnet101v2_keras'

# OPTIONAL: add a Kaggle Dataset containing ResNet101V2 (include_top=False) weights in H5 format.
# Example expected path (change to your dataset name/path):
ENCODER_WEIGHTS_PATH = None  # e.g. '/kaggle/input/resnet101v2-notop-weights/resnet101v2_notop.weights.h5'

# ImageNet normalization (keep consistent across train + inference)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def get_config(quick=QUICK_MODE):
    base = {
        'target_size': 256 if quick else 384,
        'epochs': 15 if quick else 40,
        'batch_size': 4 if quick else 6,
        'learning_rate': 1e-4 if quick else 3e-4,
        'weight_decay': 1e-4,
        'val_split': 0.1,
        'patience': 5 if quick else 8,
        'tta': 4 if quick else 8,
        'threshold': 0.4,
        'warmup_epochs': 2 if quick else 3,

        # encoder knobs
        'encoder_type': ENCODER_TYPE,
        'encoder_weights_path': ENCODER_WEIGHTS_PATH,
        'freeze_encoder': False,

        'supplemental_masks_dir': SUPPLEMENTAL_MASKS_DIR if Path(SUPPLEMENTAL_MASKS_DIR).exists() else None,
        'supplemental_images_dir': SUPPLEMENTAL_IMAGES_DIR if Path(SUPPLEMENTAL_IMAGES_DIR).exists() else None,
        'sample_submission_path': SAMPLE_SUBMISSION_PATH,
        # Must not crash during Kaggle submit runs; log errors but still write submission.csv
        'fail_on_error': False,
    }
    return base


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)
config = get_config()
print(f"Dataset root: {COMP_ROOT}")
print(f"Config: {config['target_size']}px, {config['epochs']} epochs, batch={config['batch_size']}")
print(f"Encoder: {config['encoder_type']} | weights_path={config.get('encoder_weights_path')}")

In [ ]:
# ============================================================================
# VERIFY PATHS EXIST
# ============================================================================

print("Checking paths...")
print(f"Train masks: {Path(TRAIN_MASKS_DIR).exists()} - {TRAIN_MASKS_DIR}")
print(f"Train images: {Path(TRAIN_IMAGES_DIR).exists()} - {TRAIN_IMAGES_DIR}")
print(f"Test images: {Path(TEST_IMAGES_DIR).exists()} - {TEST_IMAGES_DIR}")

# List kaggle input to help find correct paths
print("\nKaggle input contents:")
input_path = Path('/kaggle/input')
if input_path.exists():
    for p in sorted(input_path.iterdir()):
        print(f"  {p.name}/")
        if p.is_dir():
            for pp in list(p.iterdir())[:5]:
                print(f"    {pp.name}")

## Model Architecture

Custom ResNet-style encoder with U-Net decoder and attention gates.

In [ ]:
# ============================================================================
# MODEL COMPONENTS
# ============================================================================

def conv_bn_relu(x, filters, kernel_size=3, strides=1):
    """Conv2D + BatchNormalization + ReLU."""
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x


def residual_block(x, filters, out_filters, stride=1):
    """Bottleneck residual block."""
    shortcut = x
    
    # 1x1 reduce
    x = layers.Conv2D(filters, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # 3x3 conv
    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # 1x1 expand
    x = layers.Conv2D(out_filters, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    
    # Shortcut projection if needed
    if stride != 1 or shortcut.shape[-1] != out_filters:
        shortcut = layers.Conv2D(out_filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


def attention_gate(gate, skip, filters):
    """Attention gate for skip connections."""
    g = layers.Conv2D(filters, 1, use_bias=False)(gate)
    g = layers.UpSampling2D(size=2, interpolation='bilinear')(g)
    
    # Resize to match skip if needed - use Lambda layer for Keras 3 compatibility
    def resize_gate_to_skip(inputs):
        g_tensor, skip_tensor = inputs
        target_shape = keras.ops.shape(skip_tensor)[1:3]
        return keras.ops.image.resize(g_tensor, target_shape, interpolation='bilinear')
    
    g = layers.Lambda(resize_gate_to_skip)([g, skip])
    
    s = layers.Conv2D(filters, 1, use_bias=False)(skip)
    
    x = layers.Add()([g, s])
    x = layers.ReLU()(x)
    x = layers.Conv2D(1, 1, activation='sigmoid', use_bias=False)(x)
    
    return skip * x


def decoder_block(x, skip, out_filters, use_attention=True):
    """Decoder block with upsampling and skip connection."""
    x = layers.UpSampling2D(size=2, interpolation='bilinear')(x)
    
    # Resize to match skip - use Lambda layer for Keras 3 compatibility
    def resize_decoder_to_skip(inputs):
        x_tensor, skip_tensor = inputs
        target_shape = keras.ops.shape(skip_tensor)[1:3]
        return keras.ops.image.resize(x_tensor, target_shape, interpolation='bilinear')
    
    x = layers.Lambda(resize_decoder_to_skip)([x, skip])
    
    if use_attention:
        skip_filters = int(skip.shape[-1])
        skip = attention_gate(x, skip, skip_filters // 4)
    
    x = layers.Concatenate()([x, skip])
    x = conv_bn_relu(x, out_filters)
    x = conv_bn_relu(x, out_filters)
    
    return x

In [ ]:
# ============================================================================
# BUILD FULL MODEL (SELF-CONTAINED; KERAS 3 COMPATIBLE)
# ============================================================================

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet101V2


def _conv_bn_relu(x, filters, k=3):
    x = layers.Conv2D(filters, k, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)


def _resize_like(x, ref):
    # Keras 3-safe dynamic resize
    def _fn(inputs):
        x_tensor, ref_tensor = inputs
        target_hw = keras.ops.shape(ref_tensor)[1:3]
        return keras.ops.image.resize(x_tensor, target_hw, interpolation='bilinear')
    return layers.Lambda(_fn)([x, ref])


def _attention_gate(gate, skip, filters):
    g = layers.Conv2D(filters, 1, padding='same', use_bias=False)(gate)
    g = _resize_like(g, skip)

    s = layers.Conv2D(filters, 1, padding='same', use_bias=False)(skip)

    x = layers.Add()([g, s])
    x = layers.ReLU()(x)
    x = layers.Conv2D(1, 1, activation='sigmoid', use_bias=False)(x)
    return skip * x


def _decoder_block(x, skip, out_filters, use_attention=True):
    x = layers.UpSampling2D(size=2, interpolation='bilinear')(x)
    x = _resize_like(x, skip)

    if use_attention:
        att_filters = max(out_filters // 4, 8)
        skip = _attention_gate(x, skip, att_filters)

    x = layers.Concatenate()([x, skip])
    x = _conv_bn_relu(x, out_filters)
    x = _conv_bn_relu(x, out_filters)
    return x


def build_model(
    input_shape=(384, 384, 3),
    encoder_weights_path=None,
    freeze_encoder=False,
    use_attention=True,
):
    inputs = keras.Input(shape=input_shape, name='input')

    # Keras Applications encoder (no internet needed). We load weights from disk if provided.
    encoder = ResNet101V2(include_top=False, weights=None, input_tensor=inputs)

    if encoder_weights_path:
        wp = Path(encoder_weights_path)
        if wp.exists():
            print(f"Loading encoder weights from: {wp}")
            encoder.load_weights(str(wp))
        else:
            print(f"[WARN] encoder_weights_path not found: {wp} (using random init)")

    if freeze_encoder:
        encoder.trainable = False

    # Skip connections (ResNet101V2 known layer names)
    skip_layer_names = [
        'conv1_conv',
        'conv2_block3_out',
        'conv3_block4_out',
        'conv4_block23_out',
        'post_relu',
    ]

    e1, e2, e3, e4, e5 = [encoder.get_layer(n).output for n in skip_layer_names]

    d5 = _decoder_block(e5, e4, 512, use_attention=use_attention)
    d4 = _decoder_block(d5, e3, 256, use_attention=use_attention)
    d3 = _decoder_block(d4, e2, 128, use_attention=use_attention)
    d2 = _decoder_block(d3, e1, 64, use_attention=use_attention)

    d1 = layers.UpSampling2D(size=2, interpolation='bilinear')(d2)
    d1 = _conv_bn_relu(d1, 32)
    d1 = _conv_bn_relu(d1, 32)
    d1 = layers.Resizing(input_shape[0], input_shape[1], interpolation='bilinear')(d1)

    main = layers.Conv2D(1, 1, activation='sigmoid', name='main_output')(d1)

    ds4 = layers.Conv2D(1, 1, activation='sigmoid')(d5)
    ds4 = layers.Resizing(input_shape[0], input_shape[1], interpolation='bilinear', name='ds4')(ds4)

    ds3 = layers.Conv2D(1, 1, activation='sigmoid')(d4)
    ds3 = layers.Resizing(input_shape[0], input_shape[1], interpolation='bilinear', name='ds3')(ds3)

    return keras.Model(inputs=inputs, outputs=[main, ds4, ds3], name='forgery_detector')


model = build_model(
    input_shape=(config['target_size'], config['target_size'], 3),
    encoder_weights_path=config.get('encoder_weights_path'),
    freeze_encoder=config.get('freeze_encoder', False),
    use_attention=True,
)

print(f"Model built with {model.count_params():,} parameters")

# Smoke test forward pass
x = tf.zeros((1, config['target_size'], config['target_size'], 3))
out = model(x, training=False)
print(f"Output shapes: {[o.shape for o in out]}")

## Loss Functions

In [ ]:
# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

def dice_loss(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return 1.0 - (2. * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)


def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.75):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
    
    bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
    pt = tf.where(y_true > 0.5, y_pred, 1 - y_pred)
    focal_weight = tf.pow(1 - pt, gamma)
    alpha_weight = tf.where(y_true > 0.5, alpha, 1 - alpha)
    
    return tf.reduce_mean(alpha_weight * focal_weight * bce)


def bce_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
    return tf.reduce_mean(-y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred))


def combined_loss(y_true, y_pred):
    return dice_loss(y_true, y_pred) + focal_loss(y_true, y_pred) + 0.5 * bce_loss(y_true, y_pred)


def compute_total_loss(y_true, outputs):
    """Compute loss with deep supervision."""
    main_pred, ds4_pred, ds3_pred = outputs
    loss = combined_loss(y_true, main_pred)
    loss += 0.4 * dice_loss(y_true, ds4_pred)
    loss += 0.2 * dice_loss(y_true, ds3_pred)
    return loss

## Dataset

In [ ]:
# ============================================================================
# DATASET (SELF-CONTAINED; CASE_ID SPLIT + UNION MULTI-MASKS + SUPPLEMENTAL)
# ============================================================================

import re

_MASK_SUFFIX_RE = re.compile(r"^(?P<case>.+?)_(?P<idx>\d+)$")


def _mask_path_to_case_id(mask_path: str) -> str:
    stem = Path(mask_path).stem
    m = _MASK_SUFFIX_RE.match(stem)
    return m.group('case') if m else stem


def _resolve_image_path(case_id: str, train_images_dir: str, supplemental_images_dir: str | None):
    train_dir = Path(train_images_dir)
    candidates = [
        train_dir / f"{case_id}.png",
        train_dir / "forged" / f"{case_id}.png",
        train_dir / "authentic" / f"{case_id}.png",
    ]
    if supplemental_images_dir:
        sup_dir = Path(supplemental_images_dir)
        candidates += [
            sup_dir / f"{case_id}.png",
            sup_dir / "forged" / f"{case_id}.png",
            sup_dir / "authentic" / f"{case_id}.png",
        ]

    for p in candidates:
        if p.exists():
            return p
    return None


class ForgeryDataset:
    def __init__(self, items, target_size=384, augment=True, training=True):
        self.target_size = target_size
        self.augment = augment and training
        self.training = training
        # items: list[(case_id, image_path, mask_files[list[str]])]
        self.items = [(cid, str(img), [str(m) for m in masks]) for cid, img, masks in items]
        print(f"ForgeryDataset: {len(self.items)} cases (augment={augment})")

    def __len__(self):
        return len(self.items)

    def _process_mask(self, mask):
        if len(mask.shape) == 3:
            if mask.shape[0] == 1:
                mask = mask[0]
            elif mask.shape[2] == 1:
                mask = mask[:, :, 0]
            else:
                mask = mask[:, :, 0]
        if mask.max() > 1:
            mask = mask.astype(np.float32) / 255.0
        return (mask > 0.5).astype(np.float32)

    def load_sample(self, idx):
        case_id, image_path, mask_files = self.items[idx]

        image = Image.open(image_path).convert('RGB')
        orig_w, orig_h = image.size

        # Union masks, but first resize every mask to the image's native size.
        # Some cases have multiple masks saved at different resolutions.
        mask_union = None
        for mf in mask_files:
            m = np.load(mf)
            m = self._process_mask(m)

            # Ensure 2D
            if len(m.shape) != 2:
                m = np.squeeze(m)

            # Resize to (H,W) == (orig_h, orig_w) before combining
            if m.shape[0] != orig_h or m.shape[1] != orig_w:
                m_img = Image.fromarray((m * 255).astype(np.uint8), mode='L')
                m_img = m_img.resize((orig_w, orig_h), Image.NEAREST)
                m = (np.array(m_img).astype(np.float32) / 255.0)
                m = (m > 0.5).astype(np.float32)

            mask_union = m if mask_union is None else np.maximum(mask_union, m)

        if mask_union is None:
            mask_union = np.zeros((orig_h, orig_w), dtype=np.float32)

        mask = Image.fromarray((mask_union * 255).astype(np.uint8), mode='L')

        image = image.resize((self.target_size, self.target_size), Image.BILINEAR)
        mask = mask.resize((self.target_size, self.target_size), Image.NEAREST)

        if self.augment:
            image, mask = self._augment(image, mask)

        image = np.array(image).astype(np.float32) / 255.0
        image = (image - IMAGENET_MEAN) / IMAGENET_STD

        mask = np.array(mask).astype(np.float32) / 255.0
        mask = (mask > 0.5).astype(np.float32)
        mask = np.expand_dims(mask, axis=-1)

        return image, mask

    def _augment(self, image, mask):
        if random.random() < 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.5:
            image = image.transpose(Image.FLIP_TOP_BOTTOM)
            mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
        if random.random() < 0.3:
            angle = random.choice([90, 180, 270])
            image = image.rotate(angle, resample=Image.BILINEAR)
            mask = mask.rotate(angle, resample=Image.NEAREST)
        if random.random() < 0.3:
            angle = random.uniform(-25, 25)
            image = image.rotate(angle, resample=Image.BILINEAR, fillcolor=(128, 128, 128))
            mask = mask.rotate(angle, resample=Image.NEAREST, fillcolor=0)
        if random.random() < 0.5:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.75, 1.25))
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.75, 1.25))
        if random.random() < 0.2:
            image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
        return image, mask

    def generator(self):
        indices = list(range(len(self.items)))
        if self.training:
            random.shuffle(indices)
        for idx in indices:
            yield self.load_sample(idx)


def create_tf_dataset(dataset, batch_size):
    target_size = dataset.target_size
    tf_dataset = tf.data.Dataset.from_generator(
        dataset.generator,
        output_signature=(
            tf.TensorSpec(shape=(target_size, target_size, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(target_size, target_size, 1), dtype=tf.float32),
        )
    )
    if dataset.training:
        tf_dataset = tf_dataset.shuffle(1000)
    tf_dataset = tf_dataset.batch(batch_size, drop_remainder=dataset.training)
    tf_dataset = tf_dataset.prefetch(tf.data.AUTOTUNE)
    return tf_dataset


def create_datasets_with_supplemental(
    train_masks_dir,
    train_images_dir,
    target_size,
    val_split,
    supplemental_masks_dir=None,
    supplemental_images_dir=None,
):
    mask_paths = sorted([str(p) for p in Path(train_masks_dir).glob('*.npy')])
    if supplemental_masks_dir and Path(supplemental_masks_dir).exists():
        mask_paths += sorted([str(p) for p in Path(supplemental_masks_dir).glob('*.npy')])

    print(f"Found {len(mask_paths)} mask files (train+supplemental)")

    by_case = {}
    for mp in mask_paths:
        cid = _mask_path_to_case_id(mp)
        by_case.setdefault(cid, []).append(mp)

    case_ids = sorted(by_case.keys())

    items = []
    missing_images = 0
    for cid in case_ids:
        img_path = _resolve_image_path(cid, train_images_dir, supplemental_images_dir)
        if img_path is None:
            missing_images += 1
            continue
        items.append((cid, img_path, by_case[cid]))

    if missing_images:
        print(f"[WARN] {missing_images} cases skipped (no matching image found).")

    random.shuffle(items)
    val_size = int(len(items) * val_split)
    val_items = items[:val_size]
    train_items = items[val_size:]

    print(f"Train cases: {len(train_items)}, Val cases: {len(val_items)}")

    train_dataset = ForgeryDataset(train_items, target_size=target_size, augment=True, training=True)
    val_dataset = ForgeryDataset(val_items, target_size=target_size, augment=False, training=False)

    return train_dataset, val_dataset


print(f"Train masks dir: {TRAIN_MASKS_DIR}")
print(f"Train images dir: {TRAIN_IMAGES_DIR}")
print(f"Supplemental masks dir: {config.get('supplemental_masks_dir')}")
print(f"Supplemental images dir: {config.get('supplemental_images_dir')}")


## Training

In [ ]:
# ============================================================================
# TRAINING
# ============================================================================

def cosine_schedule_with_warmup(epoch, total_epochs, warmup_epochs, base_lr):
    """More conservative learning rate schedule."""
    if epoch < warmup_epochs:
        # Linear warmup - more gradual
        return base_lr * (epoch + 1) / warmup_epochs
    else:
        # Cosine decay with minimum learning rate
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        min_lr = base_lr * 0.01  # Don't decay below 1% of base LR
        return min_lr + (base_lr - min_lr) * 0.5 * (1 + np.cos(np.pi * progress))


@tf.function
def train_step(model, optimizer, images, masks):
    with tf.GradientTape() as tape:
        outputs = model(images, training=True)
        loss = compute_total_loss(masks, outputs)
    
    gradients = tape.gradient(loss, model.trainable_variables)
    # More aggressive gradient clipping for stability
    gradients, _ = tf.clip_by_global_norm(gradients, 0.5)  # Reduced from 1.0
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return loss


@tf.function
def val_step(model, images, masks, threshold=0.4):
    outputs = model(images, training=False)
    main_pred = outputs[0]
    loss = combined_loss(masks, main_pred)
    
    pred_binary = tf.cast(main_pred > threshold, tf.float32)
    masks_float = tf.cast(masks, tf.float32)
    intersection = tf.reduce_sum(pred_binary * masks_float)
    pred_sum = tf.reduce_sum(pred_binary)
    gt_sum = tf.reduce_sum(masks_float)
    
    return loss, intersection, pred_sum, gt_sum


def train_model(model, config):
    print("\n" + "=" * 60)
    print("TRAINING")
    print("=" * 60)
    
    # GPU setup
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"GPU: {gpus[0]}")
        for gpu in gpus:
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
            except:
                pass
    
    # Create datasets (split by case_id; union multi-masks; include supplemental)
    train_dataset, val_dataset = create_datasets_with_supplemental(
        train_masks_dir=TRAIN_MASKS_DIR,
        train_images_dir=TRAIN_IMAGES_DIR,
        supplemental_masks_dir=config.get('supplemental_masks_dir'),
        supplemental_images_dir=config.get('supplemental_images_dir'),
        target_size=config['target_size'],
        val_split=config['val_split'],
    )
    
    if train_dataset is None or len(train_dataset) == 0:
        print("ERROR: Could not create datasets. Check your paths!")
        return model, 0
    
    train_loader = create_tf_dataset(train_dataset, config['batch_size'])
    val_loader = create_tf_dataset(val_dataset, config['batch_size'])
    
    # Optimizer
    optimizer = keras.optimizers.AdamW(
        learning_rate=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    best_f1, best_weights = 0, None
    patience_cnt = 0
    steps_per_epoch = max(len(train_dataset) // config['batch_size'], 1)
    
    print(f"Steps per epoch: {steps_per_epoch}")
    print(f"\n{'Epoch':<6} {'Train':<10} {'Val':<10} {'F1':<10} {'LR':<12}")
    print("-" * 50)
    
    for epoch in range(config['epochs']):
        lr = cosine_schedule_with_warmup(epoch, config['epochs'], config['warmup_epochs'], config['learning_rate'])
        optimizer.learning_rate.assign(lr)
        
        # Train
        train_loss, train_batches = 0, 0
        for images, masks in tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False, total=steps_per_epoch):
            loss = train_step(model, optimizer, images, masks)
            train_loss += float(loss)
            train_batches += 1
        train_loss /= max(train_batches, 1)
        
        # Validate
        val_loss, val_batches = 0, 0
        total_intersection, total_pred, total_gt = 0, 0, 0
        for images, masks in val_loader:
            loss, intersection, pred_sum, gt_sum = val_step(model, images, masks, config['threshold'])
            val_loss += float(loss)
            total_intersection += float(intersection)
            total_pred += float(pred_sum)
            total_gt += float(gt_sum)
            val_batches += 1
        val_loss /= max(val_batches, 1)
        
        precision = total_intersection / (total_pred + 1e-8)
        recall = total_intersection / (total_gt + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8) * 100
        
        marker = ""
        if f1 > best_f1:
            best_f1 = f1
            best_weights = [w.numpy() for w in model.weights]
            patience_cnt = 0
            marker = " *"
        else:
            patience_cnt += 1
        
        # Additional metrics for debugging
        val_improvement = "↑" if val_loss < (best_f1 / 100 if best_f1 > 0 else float('inf')) else "↓"
        print(f"{epoch+1:<6} {train_loss:<10.4f} {val_loss:<10.4f} {f1:<9.2f}% {lr:<12.6f}{marker} (patience: {patience_cnt}/{config['patience']})")
        
        if patience_cnt >= config['patience']:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Restore best weights
    if best_weights:
        for w, bw in zip(model.weights, best_weights):
            w.assign(bw)

    # Threshold sweep on validation set (pick best F1 threshold)
    thresholds = config.get('threshold_sweep', [0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55])
    thr_stats = {float(t): {'intersection': 0.0, 'pred_sum': 0.0, 'gt_sum': 0.0} for t in thresholds}

    for images, masks in val_loader:
        outputs = model(images, training=False)
        pred = outputs[0] if isinstance(outputs, (list, tuple)) else outputs
        for t in thresholds:
            t = float(t)
            pred_binary = tf.cast(pred > t, tf.float32)
            thr_stats[t]['intersection'] += float(tf.reduce_sum(pred_binary * masks))
            thr_stats[t]['pred_sum'] += float(tf.reduce_sum(pred_binary))
            thr_stats[t]['gt_sum'] += float(tf.reduce_sum(masks))

    best_thr, best_thr_f1 = None, -1.0
    for t in thresholds:
        t = float(t)
        inter = thr_stats[t]['intersection']
        ps = thr_stats[t]['pred_sum']
        gs = thr_stats[t]['gt_sum']
        precision = inter / (ps + 1e-8)
        recall = inter / (gs + 1e-8)
        f1_t = 2 * precision * recall / (precision + recall + 1e-8) * 100
        if f1_t > best_thr_f1:
            best_thr_f1 = f1_t
            best_thr = t

    if best_thr is not None:
        config['threshold'] = best_thr
        print(f"\nBest threshold on val: {best_thr:.2f} (F1={best_thr_f1:.2f}%)")

    # Save
    try:
        model.save_weights(f"{OUTPUT_DIR}/model_weights.weights.h5")
        print(f"\nWeights saved to {OUTPUT_DIR}/model_weights.weights.h5")
    except Exception as e:
        print(f"Could not save weights: {e}")

    print(f"Best F1: {best_f1:.2f}%")

    return model, best_f1

## Prediction with TTA

In [ ]:
# ============================================================================
# PREDICTION WITH TTA
# ============================================================================

def predict_tta(model, image_np, tta_level=8):
    preds = []
    
    # Original
    img = np.expand_dims(image_np, axis=0)
    outputs = model(img, training=False)
    preds.append(outputs[0][0].numpy())
    
    if tta_level >= 2:
        # Horizontal flip
        img = np.expand_dims(np.fliplr(image_np), axis=0)
        outputs = model(img, training=False)
        preds.append(np.fliplr(outputs[0][0].numpy()))
    
    if tta_level >= 4:
        # Vertical flip
        img = np.expand_dims(np.flipud(image_np), axis=0)
        outputs = model(img, training=False)
        preds.append(np.flipud(outputs[0][0].numpy()))
        
        # Both flips
        img = np.expand_dims(np.flipud(np.fliplr(image_np)), axis=0)
        outputs = model(img, training=False)
        preds.append(np.flipud(np.fliplr(outputs[0][0].numpy())))
    
    if tta_level >= 8:
        # 90 and 270 degree rotations
        for k in [1, 3]:
            img = np.expand_dims(np.rot90(image_np, k=k), axis=0)
            outputs = model(img, training=False)
            preds.append(np.rot90(outputs[0][0].numpy(), k=-k))
    
    return np.mean(preds, axis=0)


def rle_encode(mask):
    """RLE encode a binary mask."""
    if len(mask.shape) == 3:
        mask = mask[:, :, 0]
    
    pixels = mask.T.flatten()
    pixels = (pixels > 0.5).astype(np.uint8)
    
    if pixels.sum() == 0:
        return "authentic"
    
    runs = []
    prev = -2
    for i in np.where(pixels == 1)[0]:
        i = int(i)
        if i > prev + 1:
            runs.extend([i + 1, 0])
        runs[-1] += 1
        prev = i
    
    return json.dumps([int(x) for x in runs])


def _index_test_images(test_dir: Path, exts=(".png", ".jpg", ".jpeg", ".tif", ".tiff")):
    idx = {}
    for ext in exts:
        for p in test_dir.rglob(f"*{ext}"):
            idx[p.stem] = p
    return idx


def generate_submission(model, config):
    """Generate submission.csv (robust for hidden test)."""
    print("\n" + "=" * 60)
    print("GENERATING SUBMISSION")
    print("=" * 60)

    test_dir = Path(TEST_IMAGES_DIR)
    if not test_dir.exists():
        print(f"Test directory not found: {TEST_IMAGES_DIR}")
        return None

    image_index = _index_test_images(test_dir)

    # Prefer sample_submission case_ids if it looks complete; otherwise fall back to all images.
    case_ids = None
    sample_path = Path(config.get('sample_submission_path', ''))
    if sample_path.exists():
        try:
            sdf = pd.read_csv(sample_path)
            if 'case_id' in sdf.columns:
                case_ids = [str(x) for x in sdf['case_id'].tolist()]
        except Exception as e:
            print(f"[WARN] Could not read sample_submission: {e}")
            case_ids = None

    if not case_ids or len(case_ids) < len(image_index):
        case_ids = sorted(image_index.keys())

    print(f"Cases: {len(case_ids)} | Images found: {len(image_index)}")
    print(f"TTA: {config['tta']}x, Threshold: {config['threshold']}")

    results = []
    errors = []

    for case_id in tqdm(case_ids, desc='Predicting'):
        img_path = image_index.get(str(case_id))
        if img_path is None:
            errors.append(f"Missing image for case_id={case_id}")
            results.append({'case_id': str(case_id), 'annotation': 'authentic'})
            continue

        try:
            image = Image.open(img_path).convert('RGB')
            original_size = image.size

            img_resized = image.resize((config['target_size'], config['target_size']), Image.BILINEAR)
            img_np = np.array(img_resized).astype(np.float32) / 255.0
            img_np = (img_np - IMAGENET_MEAN) / IMAGENET_STD

            pred = predict_tta(model, img_np, config['tta'])
            pred_resized = tf.image.resize(pred, [original_size[1], original_size[0]], method='bilinear').numpy()

            mask = (pred_resized > config['threshold']).astype(np.uint8)
            results.append({'case_id': str(case_id), 'annotation': rle_encode(mask)})

        except Exception as e:
            errors.append(f"case_id={case_id}: {e}")
            # Notebook must finish successfully to submit: fill authentic but DO log.
            results.append({'case_id': str(case_id), 'annotation': 'authentic'})

    df = pd.DataFrame(results)
    out_path = Path(OUTPUT_DIR) / 'submission.csv'
    df.to_csv(out_path, index=False)

    auth = sum(1 for r in results if r['annotation'] == 'authentic')
    print(f"\nSaved: {out_path}")
    print(f"Authentic: {auth}, Forged: {len(results) - auth}")

    if errors:
        print("\n[WARN] Submission had errors (first 20):")
        for e in errors[:20]:
            print('  -', e)
        if len(errors) > 20:
            print(f"  ... and {len(errors) - 20} more")

    return df

## Run Training

In [ ]:
# ============================================================================
# TRAIN THE MODEL
# ============================================================================

model, best_f1 = train_model(model, config)

In [ ]:
# ============================================================================
# GENERATE SUBMISSION
# ============================================================================

if Path(TEST_IMAGES_DIR).exists():
    submission_df = generate_submission(model, config)
    if submission_df is not None:
        display(submission_df.head(10))
else:
    print(f"Test images not found at {TEST_IMAGES_DIR}")

In [ ]:
print("\n" + "=" * 60)
print(f"COMPLETE! Best F1: {best_f1:.2f}%")
print("=" * 60)
print(f"\nSubmission saved to: {OUTPUT_DIR}/submission.csv")